In [1]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

from pandas import ExcelWriter

import re

import pdfplumber

import os

from bs4 import BeautifulSoup

from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.alert import Alert

from googletrans import Translator



In [2]:
# %%

#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'CN CSRC'

print(f"Running {regulatorName} Web Scraping Tool v.1.1")

now=datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - moodys.com\\Desktop\\Regulator\\{regulatorName}" ## to comment for the local environment

#scriptfolder=os.path.dirname(os.path.abspath(__file__))

os.chdir(scriptfolder)

writer = ExcelWriter(filename)

tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process



if os.path.exists(tempfolder):

    for rem in os.listdir(tempfolder):

        os.remove(os.path.join(tempfolder, rem))

else:

    os.mkdir(tempfolder)



Running CN CSRC Web Scraping Tool v.1.1


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------



#Starting Chrome driver, set to download files in tempfolder
#Try to download the insecure file in 



chrome_options = Options()
#chrome_options.add_argument("--window-size=1920,1080")
chrome_options.add_argument("--allow-running-insecure-content")  # Allow insecure content

chrome_options.add_experimental_option("prefs", {
    "download.default_directory": tempfolder,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
})



In [4]:
#------------------------------------------------ Begin_Fouction ----------------------------------------


def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


# Initialize the translator
translator = Translator()
def translate_text(text):
    return translator.translate(text, src='zh-cn', dest='en').text

In [5]:
# %%

#------------------------------------------------ Begin_Variable ----------------------------------------

regdict = { 'CN CSRC 1': 'http://www.csrc.gov.cn/',
            'CN CSRC 2': 'http://www.csrc.gov.cn/',
            'CN CSRC 3': 'http://www.csrc.gov.cn/',
            'CN CSRC 4': 'http://www.csrc.gov.cn/',
            'CN CSRC 5': 'http://www.csrc.gov.cn/',
            }

Typology ={

            'CN CSRC 1': 'List of Securities Companies',
            'CN CSRC 2': 'List of Futures Companies',
            'CN CSRC 3': 'List of Fund Management Companies',

            'CN CSRC 4': 'List of QFIIs',
            'CN CSRC 5': 'List of Custodian Banks for Qualified Foreign Investors',

}

sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'Name_2':[],'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
         'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
         'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
         'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
         'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
         'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

In [ ]:
# %%

#------------------------------------------------ Begin_Main ----------------------------------------

for reg in regdict:
    
    driver = webdriver.Chrome(options=chrome_options)
    driver.maximize_window()
    sleep(2)
    print(f'Working with list {reg}')
    driver.get(regdict[reg])
    
    input_element = driver.find_element(By.ID, 'searchWord')
    # Enter the search term
    if reg == 'CN CSRC 1' or reg == 'CN CSRC 3':
        if reg ==     'CN CSRC 1':
            input_element.send_keys('证券公司')
        elif reg ==     'CN CSRC 3':
            input_element.send_keys('基金管理机构')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        newsInfo = soup.find('div',{'class':'wordGuide Residence-permit'}) 
        sleep(2)
        inner = newsInfo.find('a')
        
        if  '名录' in inner['data'] :
            print(inner['href'])
            chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+inner['href'])  # Replace example.com with your site's domain
            driver = webdriver.Chrome(options=chrome_options)
            driver.get(inner['href'])
            sleep(3)
        print('Try to Search Documents in the news information list')
        soup2 = BeautifulSoup(driver.page_source, 'html.parser')  
        sleep(2)
        driver.find_element(By.XPATH,'//*[@id="files"]').click()
        print('Download the file')
        sleep(2)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        dataframe = dataframe.iloc[:,-2:]

        # Apply the translation function to each cell in the dataframe
        # 5 min about 150 data of two columns
        translated_dataframe = pd.DataFrame()
        translated_dataframe['CompanyName'] = dataframe['公司名称'].map(translate_text)
        translated_dataframe['RegistedCity'] =  dataframe['辖区（注册地）'].map(translate_text)
        translated_dataframe['CompanyNameCN'] = dataframe['公司名称']
        #translated_dataframe = dataframe.map(translate_text)
        #translated_dataframe =  translated_dataframe.rename(columns={"公司名称": "CompanyName", "辖区（注册地）": "Registed City"})
        for name,city,cname in zip(translated_dataframe['CompanyName'],translated_dataframe['RegistedCity'],translated_dataframe['CompanyNameCN']):
            sqldict['Name'].append(name)
            sqldict['Name_2'].append(cname)
            sqldict['City'].append(city)
            sqldict['ListProcessDate'].append(processdate)    
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
        driver.quit()
    
    elif  reg == 'CN CSRC 2': 
        input_element.send_keys('期货公司')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        sleep(2)
        soup = BeautifulSoup(driver.page_source, 'html.parser')  
        newsInfo = soup.find('div',{'class':'newsInfo'}) 
        innerlinks = newsInfo.find('ul').find_all('a')
        for inner in innerlinks:
            if '名录' in inner['data'] :
                print(inner['href'])
                chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+inner['href'])  # Replace example.com with your site's domain
                driver = webdriver.Chrome(options=chrome_options)
                driver.get(inner['href'])
                sleep(3)

        print('Try to Search Documents in the news information list')
        soup2 = BeautifulSoup(driver.page_source, 'html.parser')  
        driver.find_element(By.XPATH,'//*[@id="files"]').click()
        print('Download the file')
        sleep(2)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        dataframe = dataframe.iloc[:,-2:]
        dataframe.replace('NaN', pd.NA, inplace=True)
        # Forward fill the NaN values
        dataframe['辖区'].fillna(method='ffill', inplace=True)
        #print(dataframe)
        # Apply the translation function to each cell in the dataframe
        # 5 min about 150 data of two columns
        translated_dataframe = pd.DataFrame()
        translated_dataframe['CompanyNameCN'] = dataframe['期货公司名称']
        translated_dataframe['CompanyName'] = dataframe['期货公司名称'].map(translate_text)
        translated_dataframe['RegistedCity'] =  dataframe['辖区'].map(translate_text)
        #translated_dataframe = dataframe.map(translate_text)
        #translated_dataframe =  translated_dataframe.rename(columns={"辖区": "Registed City","期货公司名称": "CompanyName", })
        for name,city,cname in zip(translated_dataframe['CompanyName'],translated_dataframe['RegistedCity'],translated_dataframe['CompanyNameCN']):
            sqldict['Name'].append(name)
            sqldict['Name_2'].append(cname)
            sqldict['City'].append(city)
            sqldict['ListProcessDate'].append(processdate)    
            sqldict['RegulationType'].append('Regulated')
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['ListName'].append(Typology[reg])
        sqldict = bourange_same_length_array(sqldict)
        driver.quit()

    elif reg == 'CN CSRC 4' or reg == 'CN CSRC 5':
        if reg == 'CN CSRC 4':
            input_element.send_keys('合格境外投资者')
        elif reg == 'CN CSRC 5':
            input_element.send_keys('合格境外投资者托管行')
        print('Input Chinese Keywords ')
        # Submit the form
        input_element.send_keys(Keys.RETURN)
        sleep(3)
        window_handles = driver.window_handles
        if len(window_handles)>1:
            driver.switch_to.window(window_handles[1])
        else:
            print('Can not open the second Tab')    
        sleep(2)
        try:
            soup = BeautifulSoup(driver.page_source, 'html.parser')  
            sleep(2)
            wordguide = soup.find('div',{'class':'wordGuide Residence-permit'} )
            sleep(2)
            innerlink = wordguide.find('a')['href']
        except:
            driver.refresh()
            sleep(2)
            soup = BeautifulSoup(driver.page_source, 'html.parser')  
            sleep(2)
            wordguide = soup.find('div',{'class':'wordGuide Residence-permit'} )
            sleep(2)
            innerlink = wordguide.find('a')['href']
            
        chrome_options.add_argument("--unsafely-treat-insecure-origin-as-secure="+innerlink)  # Replace example.com with your site's domain
        driver = webdriver.Chrome(options=chrome_options)
        driver.get(innerlink)
        sleep(3)
        print('Try to Search Documents in the news information list')
        soup2 = BeautifulSoup(driver.page_source, 'html.parser')  
        driver.find_element(By.XPATH,'//*[@id="files"]').click()
        print('Download the file')
        sleep(2)
        dl_files = [os.path.join(tempfolder, f) for f in os.listdir(tempfolder)]
        sleep(2)
        dataframe = pd.read_excel(dl_files[0], engine='xlrd')
        if reg == 'CN CSRC 4':
            dataframe.columns = dataframe.iloc[0]
            dataframe = dataframe[1:].iloc[:,1:]
            dataframe['英文名称'].fillna('', inplace=True)
            for ENname,CNname,city,ApproveDate in zip(dataframe['英文名称'],dataframe['中文名称'],dataframe['注册地'],dataframe['批准日期']):
                if CNname!='':
                    sqldict['ListProcessDate'].append(processdate)    
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['ListName'].append(Typology[reg])
                    translate_cntry = translate_text(city)
                    sqldict['Cntry'].append(translate_cntry)
                    sqldict['RegulationDate'].append(ApproveDate)
                    if ENname=='':
                        #print(ENname,CNname,city,ApproveDate)
                        translate_name = translate_text(CNname)
                        sqldict['Name'].append(translate_name)
                        sqldict['Name_2'].append(CNname)
                        #print(translate_name)
                    else:
                        sqldict['Name'].append(ENname)
                        sqldict['Name_2'].append(CNname)
            sqldict = bourange_same_length_array(sqldict)
            driver.quit()
        elif reg == 'CN CSRC 5':
            for info,cname in zip(dataframe['合格境外投资者托管行英文名称'],dataframe['合格境外投资者托管行中文名称']):
                #print(info)
                sqldict['Name'].append(info)
                sqldict['Name_2'].append(cname)
                sqldict['ListProcessDate'].append(processdate)    
                sqldict['RegulationType'].append('Regulated')
                sqldict['RegCtry'].append(reg.split()[0])
                sqldict['RegCode'].append(reg.split()[1])
                sqldict['ListCode'].append(reg.split()[2])
                sqldict['ListName'].append(Typology[reg])
            sqldict = bourange_same_length_array(sqldict)
            driver.quit()
                
                        
    
    if os.path.exists(tempfolder):

        for rem in os.listdir(tempfolder):

            os.remove(os.path.join(tempfolder, rem))
    



Working with list CN CSRC 1
Input Chinese Keywords 
http://www.csrc.gov.cn/csrc/c101900/c1029659/content.shtml
Try to Search Documents in the news information list
Download the file
Working with list CN CSRC 2
Input Chinese Keywords 
http://www.csrc.gov.cn/csrc/c101920/c1039268/content.shtml
Try to Search Documents in the news information list
Download the file


C:\Users\wuj1\AppData\Local\Temp\9\ipykernel_16072\578554458.py:105: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dataframe['辖区'].fillna(method='ffill', inplace=True)
C:\Users\wuj1\AppData\Local\Temp\9\ipykernel_16072\578554458.py:105: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  dataframe['辖区'].fillna(method='ffill', inplace=True)


Working with list CN CSRC 3
Input Chinese Keywords 
http://www.csrc.gov.cn/csrc/c101900/c1029657/content.shtml
Try to Search Documents in the news information list
Download the file
Working with list CN CSRC 4
Input Chinese Keywords 
Try to Search Documents in the news information list
Download the file


C:\Users\wuj1\AppData\Local\Temp\9\ipykernel_16072\578554458.py:172: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  dataframe['英文名称'].fillna('', inplace=True)


Working with list CN CSRC 5
Input Chinese Keywords 
Try to Search Documents in the news information list
Download the file


IndexError: list index out of range

In [7]:
# check which is different

key_value_counts = {key: len(values) for key, values in sqldict.items()}

# Print the counts
for key, count in key_value_counts.items():
    print(f"Key '{key}' has {count} values.")

Key 'bvdid' has 1306 values.
Key 'priority' has 1306 values.
Key 'ListLabel' has 1306 values.
Key 'Typology' has 1306 values.
Key 'EntryType' has 1306 values.
Key 'Name' has 1306 values.
Key 'Name_2' has 1306 values.
Key 'InternalID_1' has 1306 values.
Key 'InternalID_1_type' has 1306 values.
Key 'InternalID_2' has 1306 values.
Key 'InternalID_2_type' has 1306 values.
Key 'InternalID_3' has 1306 values.
Key 'InternalID_3_type' has 1306 values.
Key 'CoType' has 1306 values.
Key 'License_Type' has 1306 values.
Key 'Address_1' has 1306 values.
Key 'Address_2' has 1306 values.
Key 'City' has 1306 values.
Key 'Zip' has 1306 values.
Key 'Cntry' has 1306 values.
Key 'Phone' has 1306 values.
Key 'Fax' has 1306 values.
Key 'Website' has 1306 values.
Key 'Email' has 1306 values.
Key 'RegulationType' has 1306 values.
Key 'RegulationTypeCode' has 1306 values.
Key 'RegulationDate' has 1306 values.
Key 'CancellationDate' has 1306 values.
Key 'RegCtry' has 1306 values.
Key 'RegCode' has 1306 values.


In [8]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)

df = df.drop_duplicates()

df.to_excel(writer, 'SQL Ready', index=False)

writer.save()

writer.close()

driver.quit()

sleep(3)

C:\Users\wuj1\AppData\Local\Temp\9\ipykernel_16072\3068940208.py:11: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  df.to_excel(writer, 'SQL Ready', index=False)


AttributeError: 'OpenpyxlWriter' object has no attribute 'save'

In [10]:
cname

'苏新基金管理有限公司'

In [9]:
df.to_csv('total_CN.csv')

In [ ]:

import sys
sys.stdout.reconfigure(encoding='utf-8')